# 06 - Foundry IQ knowledge base: Blob + SharePoint + Web IQ

Build **one knowledge base** that answers a question from three very different places, using direct REST calls and no Vision skills. Foundry IQ is the knowledge layer on top of Azure AI Search: knowledge sources are reusable objects, and a knowledge base combines them with a model that plans the queries.

| Knowledge source | Kind | Content | How it is read |
|---|---|---|---|
| Blob Storage | `azureBlob` | `data/search` documents you upload | **Indexed**: Search generates a data source, skillset, index and indexer |
| SharePoint | `indexedSharePoint` (or `remoteSharePoint`) | Your test site's Documents library | **Indexed** with document-level permissions, or **remote** with no index |
| Web IQ | `mcpServer` | The public web through `https://api.microsoft.ai/v3/mcp` | **Remote**: live tool calls at query time |

Web IQ is not the built-in `web` knowledge source (that one uses Bing). It joins the knowledge base through the **MCP server** knowledge source kind.

**Run order:** run every section in order. Sections 3-4 create billable indexing work; cleanup is opt-in at the end.

## Prerequisites

- **Redeploy the infrastructure** once with `scripts\deploy.ps1`. It adds a text-embedding deployment to Foundry and lets the Search managed identity call Foundry models, which the indexed knowledge sources and the knowledge base's query planner need. `.env.azure` must then contain `AZURE_AI_EMBEDDING_DEPLOYMENT_NAME`.
- Your user needs Search Service Contributor, Search Index Data Reader or Contributor and Storage Blob Data Contributor on the workshop container. `scripts\grant-workshop-access.ps1` supplies these.
- A Web IQ profile API key in `.env` as `WEBIQ_API_KEY` (notebook 01). This notebook sends it with each retrieve request and **never stores it in Search**.
- Add `SHAREPOINT_SITE_URL` and a unique `WORKSHOP_PREFIX` to `.env`. For indexed SharePoint, the facilitator must have run `scripts\setup-sharepoint-app.ps1` (see notebook 04) so `SHAREPOINT_APP_CLIENT_ID` and `SHAREPOINT_SEARCH_MI_CLIENT_ID` exist. Remote SharePoint needs no app but needs a Microsoft 365 Copilot license or Retrieval API pay-as-you-go for each querying user (notebook 05).
- Upload the two synthetic documents in `data/search` to the test site's Documents library, as in notebook 04.

All of this uses preview APIs (`2026-08-01-preview`); check the linked documentation before class.

In [ ]:
import json
import os
import re
import time
from datetime import datetime, timezone
from email.utils import format_datetime
from pathlib import Path
from urllib.parse import quote, urlparse
from uuid import UUID

import requests
from azure.identity import AzureCliCredential, get_bearer_token_provider
from dotenv import load_dotenv

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'infra' / 'main.bicep').is_file():
    raise RuntimeError('Open from the workshop root or notebooks directory.')
load_dotenv(root / '.env')
load_dotenv(root / '.env.azure')
required = ('AZURE_TENANT_ID', 'AZURE_SEARCH_ENDPOINT', 'AZURE_STORAGE_RESOURCE_ID',
            'AZURE_STORAGE_BLOB_ENDPOINT', 'AZURE_STORAGE_CONTAINER_NAME', 'AZURE_AI_ACCOUNT_NAME',
            'AZURE_AI_MODEL_DEPLOYMENT_NAME', 'AZURE_AI_EMBEDDING_DEPLOYMENT_NAME',
            'WEBIQ_API_KEY', 'SHAREPOINT_SITE_URL')
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f'Missing settings: {missing}. Redeploy and see .env.example.')
tenant_id = str(UUID(os.environ['AZURE_TENANT_ID']))
prefix = os.getenv('WORKSHOP_PREFIX') or input('Unique attendee prefix (e.g. alice01): ').strip()
if not re.fullmatch(r'[a-z][a-z0-9]{2,19}', prefix):
    raise ValueError('Use 3-20 lowercase letters/digits, starting with a letter.')
site_url = os.environ['SHAREPOINT_SITE_URL'].rstrip('/')
site = urlparse(site_url)
if (site.scheme != 'https' or not site.hostname or not site.hostname.endswith('.sharepoint.com')
        or site.username or site.password or site.port not in (None, 443)
        or site.query or site.fragment or any(c in site_url for c in ';"\r\n')):
    raise ValueError('Use a public-cloud SharePoint HTTPS site URL, not a file or sharing-link URL.')

search_endpoint = os.environ['AZURE_SEARCH_ENDPOINT'].rstrip('/')
storage_endpoint = os.environ['AZURE_STORAGE_BLOB_ENDPOINT'].rstrip('/')
storage_id = os.environ['AZURE_STORAGE_RESOURCE_ID']
container_name = os.environ['AZURE_STORAGE_CONTAINER_NAME']
aoai_endpoint = f"https://{os.environ['AZURE_AI_ACCOUNT_NAME']}.openai.azure.com"
chat_deployment = os.environ['AZURE_AI_MODEL_DEPLOYMENT_NAME']
chat_model = os.getenv('AZURE_AI_MODEL_NAME', 'gpt-5-mini')
embedding_deployment = os.environ['AZURE_AI_EMBEDDING_DEPLOYMENT_NAME']
embedding_model = os.getenv('AZURE_AI_EMBEDDING_MODEL_NAME', 'text-embedding-3-small')
webiq_api_key = os.environ['WEBIQ_API_KEY']
webiq_tools = [tool.strip() for tool in os.getenv('WEBIQ_KB_TOOLS', 'web').split(',') if tool.strip()]

api_version = '2026-08-01-preview'
blob_folder = f'{prefix}/kb'
blob_source_name = f'{prefix}-kb-blob'
sharepoint_source_name = f'{prefix}-kb-sharepoint'
webiq_source_name = f'{prefix}-kb-webiq'
knowledge_base_name = f'{prefix}-kb'

credential = AzureCliCredential(tenant_id=tenant_id)
search_token = get_bearer_token_provider(credential, 'https://search.azure.com/.default')
storage_token = get_bearer_token_provider(credential, 'https://storage.azure.com/.default')
print('Search:', search_endpoint)
print('Models:', aoai_endpoint, '|', chat_deployment, '+', embedding_deployment)
print('Knowledge base:', knowledge_base_name)

## 1. Blob Storage knowledge source

Upload the two synthetic documents to a per-attendee folder. The knowledge source points at that folder only, so attendees sharing the container do not index each other's files. Your Entra identity writes the blobs; no keys or SAS tokens are used.

In [ ]:
sample_files = sorted((root / 'data' / 'search').glob('*.txt'))
if len(sample_files) != 2:
    raise RuntimeError('Expected the two workshop sample documents under data/search.')
for path in sample_files:
    blob_name = f'{blob_folder}/{path.name}'
    response = requests.put(
        f'{storage_endpoint}/{container_name}/{quote(blob_name, safe="/")}',
        headers={
            'Authorization': f'Bearer {storage_token()}',
            'x-ms-version': '2023-11-03',
            'x-ms-date': format_datetime(datetime.now(timezone.utc), usegmt=True),
            'x-ms-blob-type': 'BlockBlob',
            'Content-Type': 'text/plain; charset=utf-8',
        },
        data=path.read_bytes(), timeout=(10, 90),
    )
    response.raise_for_status()
    print('Uploaded:', blob_name)

### Create the blob knowledge source

One PUT replaces the data source, skillset, index and indexer you built by hand in notebook 03: Search generates and runs them for you. The Search managed identity reads the blobs (`ResourceId=` connection string, no secret) and calls the embedding deployment. We use `contentExtractionMode: minimal` and turn off image verbalization, so no chat model, Content Understanding or Vision service is involved at ingestion.

In [ ]:
blob_source = {
    'name': blob_source_name, 'kind': 'azureBlob',
    'description': 'Workshop policy documents stored in Blob Storage.',
    'azureBlobParameters': {
        'connectionString': f'ResourceId={storage_id};',
        'containerName': container_name,
        'folderPath': blob_folder,
        'isADLSGen2': False,
        'ingestionParameters': {
            'networkAccessMode': 'public',
            'identity': None,
            'disableImageVerbalization': True,
            'chatCompletionModel': None,
            'embeddingModel': {
                'kind': 'azureOpenAI',
                'azureOpenAIParameters': {
                    'resourceUri': aoai_endpoint,
                    'deploymentId': embedding_deployment,
                    'modelName': embedding_model,
                },
            },
            'contentExtractionMode': 'minimal',
            'ingestionSchedule': None,
        },
    },
}
response = requests.put(
    f'{search_endpoint}/knowledgesources/{blob_source_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=blob_source, timeout=(10, 120),
)
response.raise_for_status()
print('Blob knowledge source ready:', blob_source_name)

## 2. SharePoint knowledge source

Choose how Search reads SharePoint:

- **`indexed`** copies chunks into a Search index with Entra user/group permissions (`userIds`, `groupIds`) attached to each chunk. It uses the facilitator's app registration and the Search managed-identity federated trust from notebook 04, with no client secret.
- **`remote`** queries SharePoint live as the signed-in user, with no index, no app registration and a per-user Copilot license or Retrieval API pay-as-you-go requirement (notebook 05).

Either way, the retrieve call in section 5 sends your token in `x-ms-query-source-authorization`, so Search can apply SharePoint permissions as you.

In [ ]:
SHAREPOINT_KB_MODE = 'indexed'  # 'indexed' or 'remote'

if SHAREPOINT_KB_MODE == 'indexed':
    sp_app_client_id = str(UUID(os.environ['SHAREPOINT_APP_CLIENT_ID']))
    sp_mi_client_id = str(UUID(os.environ['SHAREPOINT_SEARCH_MI_CLIENT_ID']))
    if sp_app_client_id == sp_mi_client_id:
        raise ValueError('The ingestion app and Search managed identity must have different client IDs.')
    sharepoint_kind = 'indexedSharePoint'
    sharepoint_source = {
        'name': sharepoint_source_name, 'kind': sharepoint_kind,
        'description': 'Workshop SharePoint Documents library, indexed with permissions.',
        'indexedSharePointParameters': {
            'connectionString': (
                f'SharePointOnlineEndpoint={site_url};ApplicationId={sp_app_client_id};'
                f'FederatedCredentialApplicationId={sp_mi_client_id};TenantId={tenant_id}'
            ),
            'containerName': 'defaultSiteLibrary',
            'ingestionParameters': {
                'networkAccessMode': 'public',
                'identity': None,
                'disableImageVerbalization': True,
                'chatCompletionModel': None,
                'embeddingModel': {
                    'kind': 'azureOpenAI',
                    'azureOpenAIParameters': {
                        'resourceUri': aoai_endpoint,
                        'deploymentId': embedding_deployment,
                        'modelName': embedding_model,
                    },
                },
                'contentExtractionMode': 'minimal',
                'ingestionPermissionOptions': ['userIds', 'groupIds'],
                'ingestionSchedule': None,
            },
        },
    }
elif SHAREPOINT_KB_MODE == 'remote':
    sharepoint_kind = 'remoteSharePoint'
    sharepoint_source = {
        'name': sharepoint_source_name, 'kind': sharepoint_kind,
        'description': 'Workshop SharePoint retrieved live as the signed-in user.',
        'remoteSharePointParameters': {
            'filterExpression': f'Path:"{site_url}"',
            'resourceMetadata': ['Author', 'Title'],
        },
    }
else:
    raise ValueError("SHAREPOINT_KB_MODE must be 'indexed' or 'remote'.")
response = requests.put(
    f'{search_endpoint}/knowledgesources/{sharepoint_source_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=sharepoint_source, timeout=(10, 120),
)
response.raise_for_status()
print(f'SharePoint knowledge source ready ({SHAREPOINT_KB_MODE}):', sharepoint_source_name)

## 3. Web IQ knowledge source (MCP server)

Web IQ is a remote knowledge source: Search calls its MCP tools at query time and ranks the results next to your private content. The model in the knowledge base chooses which tool to call and writes its arguments.

**The Web IQ key is deliberately not stored here.** The `authentication` block is empty, and section 5 passes `x-apikey` as a per-request header. That keeps each attendee's personal profile key out of the shared Search service. (`storedHeaders` would store one shared key instead, which would also share its rate limits.)

Preview limits for MCP knowledge sources: `minimal` reasoning effort and `alwaysQuerySource` are not supported, and tool calls can be slow, so the retrieve request sets `maxRuntimeInSeconds`. The tools listed must be ones your Web IQ profile is entitled to; set `WEBIQ_KB_TOOLS` (comma separated) to change the default `web`.

In [ ]:
webiq_source = {
    'name': webiq_source_name, 'kind': 'mcpServer',
    'description': 'Web IQ web grounding for current public information.',
    'resultsProcessing': 'rerank',
    'mcpServerParameters': {
        'serverURL': 'https://api.microsoft.ai/v3/mcp',
        'authentication': None,
        'tools': [
            {'name': tool, 'outputParsing': {'kind': 'auto'}, 'maxOutputTokens': 2000}
            for tool in webiq_tools
        ],
    },
}
response = requests.put(
    f'{search_endpoint}/knowledgesources/{webiq_source_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=webiq_source, timeout=(10, 120),
)
response.raise_for_status()
print('Web IQ knowledge source ready:', webiq_source_name, '| tools:', webiq_tools)

## 4. Wait for the indexed sources to finish ingesting

Creating an indexed knowledge source starts its generated indexer. Read each source's `createdResources`, then poll those indexers until they succeed. Remote sources (Web IQ, remote SharePoint) have nothing to wait for. Do not query until this finishes, or the knowledge base will answer from partial content.

In [ ]:
pending = {}
for source_name in (blob_source_name, sharepoint_source_name):
    response = requests.get(
        f'{search_endpoint}/knowledgesources/{source_name}?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
    )
    response.raise_for_status()
    definition = response.json()
    parameters = definition.get('azureBlobParameters') or definition.get('indexedSharePointParameters') or {}
    created = parameters.get('createdResources') or {}
    if created.get('indexer'):
        pending[source_name] = created['indexer']
print('Generated indexers:', pending or 'none (remote only)')

for attempt in range(90):
    for source_name, indexer_name in list(pending.items()):
        response = requests.get(
            f'{search_endpoint}/indexers/{indexer_name}/status?api-version={api_version}',
            headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
        )
        response.raise_for_status()
        last_run = response.json().get('lastResult') or {}
        status = last_run.get('status')
        if status in ('transientFailure', 'persistentFailure') or last_run.get('errors') or last_run.get('itemsFailed'):
            raise RuntimeError(f'{source_name} ingestion failed: {json.dumps(last_run, indent=2)}')
        if status == 'success':
            for warning in last_run.get('warnings') or []:
                print('Warning:', warning.get('message'))
            print(f'{source_name}: {last_run.get("itemsProcessed", 0)} documents indexed.')
            del pending[source_name]
    if not pending:
        break
    time.sleep(10)
else:
    raise TimeoutError(f'Ingestion is still running for: {list(pending)}')

## 5. Create the knowledge base

The knowledge base lists the three sources and the model that plans retrieval. For each question the model decides which sources are worth querying, writes a subquery per source, and Search merges and reranks everything into one result. With `answerSynthesis`, the same model then writes a cited answer from the retrieved evidence.

- `retrievalInstructions` tells the planner what each source is for. Clear instructions matter more than anything else here.
- `retrievalReasoningEffort` is `low`: the Web IQ (MCP) source does not support `minimal`.
- The model is a Foundry deployment that the Search managed identity calls (Cognitive Services User); no model keys are used.

In [ ]:
knowledge_base = {
    'name': knowledge_base_name,
    'description': 'Workshop policies (Blob and SharePoint) plus live public web information from Web IQ.',
    'retrievalInstructions': (
        f'Use {blob_source_name} and {sharepoint_source_name} for questions about internal company policy, '
        f'travel, onboarding or documents. Use {webiq_source_name} only for current public information, '
        'external facts or comparisons with the outside world.'
    ),
    'answerInstructions': (
        'Answer concisely and cite every source. Say which facts come from internal documents and which '
        'come from the web. If the sources do not contain the answer, say you do not know.'
    ),
    'outputMode': 'answerSynthesis',
    'knowledgeSources': [
        {'name': blob_source_name},
        {'name': sharepoint_source_name},
        {'name': webiq_source_name},
    ],
    'models': [{
        'kind': 'azureOpenAI',
        'azureOpenAIParameters': {
            'resourceUri': aoai_endpoint,
            'deploymentId': chat_deployment,
            'modelName': chat_model,
        },
    }],
    'retrievalReasoningEffort': {'kind': 'low'},
}
response = requests.put(
    f'{search_endpoint}/knowledgebases/{knowledge_base_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=knowledge_base, timeout=(10, 90),
)
response.raise_for_status()
print('Knowledge base ready:', knowledge_base_name)

## 6. Ask questions

Each request carries three kinds of identity:

- `Authorization`: your token for Azure AI Search.
- `x-ms-query-source-authorization`: your token again, so SharePoint permissions are evaluated as **you**, not as the service.
- `<web-iq-source>-header-name` / `-header-value`: the Web IQ key, forwarded only to the MCP server for this request.

`includeActivity` returns what each source did, so you can see which sources the planner actually used. The three questions are expected to route differently; compare the activity for each, and treat routing as the model's judgment, not a guarantee.

### 6a. An internal question

In [ ]:
question = 'What receipts do I need to submit for travel reimbursement?'
response = requests.post(
    f'{search_endpoint}/knowledgebases/{knowledge_base_name}/retrieve?api-version={api_version}',
    headers={
        'Authorization': f'Bearer {search_token()}',
        'x-ms-query-source-authorization': search_token(),
        f'{webiq_source_name}-header-name': 'x-apikey',
        f'{webiq_source_name}-header-value': webiq_api_key,
    },
    json={
        'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}],
        'includeActivity': True,
        'maxRuntimeInSeconds': 60,
        'knowledgeSourceParams': [
            {'knowledgeSourceName': blob_source_name, 'kind': 'azureBlob',
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': sharepoint_source_name, 'kind': sharepoint_kind,
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': webiq_source_name, 'kind': 'mcpServer', 'includeReferences': True},
        ],
    }, timeout=(10, 120),
)
response.raise_for_status()
result = response.json()
for message in result.get('response', []):
    for part in message.get('content', []):
        print(part.get('text'))
print('\nReferences:')
for reference in result.get('references', []):
    print('-', reference.get('type'), reference.get('id'), reference.get('docKey') or '')
print('\nActivity:')
for step in result.get('activity', []):
    print('-', step.get('type'), step.get('knowledgeSourceName', ''), step.get('count', ''), step.get('error') or '')
if response.status_code == 206 or any(step.get('error') for step in result.get('activity', [])):
    raise RuntimeError('Retrieval was partial or a source failed. Inspect the activity output above.')

### 6b. A question only the web can answer

Internal documents cannot answer this. A good run shows the Web IQ source used and the internal sources skipped or returning nothing relevant.

In [ ]:
question = 'What is the latest stable release of Python, and when was it released?'
response = requests.post(
    f'{search_endpoint}/knowledgebases/{knowledge_base_name}/retrieve?api-version={api_version}',
    headers={
        'Authorization': f'Bearer {search_token()}',
        'x-ms-query-source-authorization': search_token(),
        f'{webiq_source_name}-header-name': 'x-apikey',
        f'{webiq_source_name}-header-value': webiq_api_key,
    },
    json={
        'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}],
        'includeActivity': True,
        'maxRuntimeInSeconds': 60,
        'knowledgeSourceParams': [
            {'knowledgeSourceName': blob_source_name, 'kind': 'azureBlob',
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': sharepoint_source_name, 'kind': sharepoint_kind,
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': webiq_source_name, 'kind': 'mcpServer', 'includeReferences': True},
        ],
    }, timeout=(10, 120),
)
response.raise_for_status()
result = response.json()
for message in result.get('response', []):
    for part in message.get('content', []):
        print(part.get('text'))
print('\nReferences:')
for reference in result.get('references', []):
    print('-', reference.get('type'), reference.get('id'), reference.get('docKey') or '')
print('\nActivity:')
for step in result.get('activity', []):
    print('-', step.get('type'), step.get('knowledgeSourceName', ''), step.get('count', ''), step.get('error') or '')
if response.status_code == 206 or any(step.get('error') for step in result.get('activity', [])):
    raise RuntimeError('Retrieval was partial or a source failed. Inspect the activity output above.')

### 6c. A question that needs both

This is the point of one knowledge base: the planner should query internal documents **and** the web, and the answer should separate the two.

In [ ]:
question = ('How does our travel reimbursement policy compare with common corporate travel '
            'expense best practices on the public web?')
response = requests.post(
    f'{search_endpoint}/knowledgebases/{knowledge_base_name}/retrieve?api-version={api_version}',
    headers={
        'Authorization': f'Bearer {search_token()}',
        'x-ms-query-source-authorization': search_token(),
        f'{webiq_source_name}-header-name': 'x-apikey',
        f'{webiq_source_name}-header-value': webiq_api_key,
    },
    json={
        'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}],
        'includeActivity': True,
        'maxRuntimeInSeconds': 90,
        'knowledgeSourceParams': [
            {'knowledgeSourceName': blob_source_name, 'kind': 'azureBlob',
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': sharepoint_source_name, 'kind': sharepoint_kind,
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': webiq_source_name, 'kind': 'mcpServer', 'includeReferences': True},
        ],
    }, timeout=(10, 150),
)
response.raise_for_status()
result = response.json()
for message in result.get('response', []):
    for part in message.get('content', []):
        print(part.get('text'))
print('\nReferences:')
for reference in result.get('references', []):
    print('-', reference.get('type'), reference.get('id'), reference.get('docKey') or '')
print('\nActivity:')
for step in result.get('activity', []):
    print('-', step.get('type'), step.get('knowledgeSourceName', ''), step.get('count', ''), step.get('error') or '')
if response.status_code == 206 or any(step.get('error') for step in result.get('activity', [])):
    raise RuntimeError('Retrieval was partial or a source failed. Inspect the activity output above.')

## 7. Evaluate, do not just read the answer

- A fluent answer is not proof. Open `references` and confirm every claim traces to a blob, SharePoint item or web result.
- Ask the same internal question as two users with different SharePoint access (indexed mode applies ACLs from the chunk metadata; remote mode applies Microsoft 365 access). Blob content here has no per-user ACLs: everyone with the container role sees it.
- Web results change over time and depend on your Web IQ entitlements and rate limits.
- Compare the activity for 6a, 6b and 6c. If the planner queried a source it should not have, tighten `retrievalInstructions` and re-run section 5.

## 8. Optional cleanup

Enable the next cell to delete the knowledge base, then the three knowledge sources (a source cannot be deleted while a knowledge base references it). Missing objects (404) are allowed; other errors are not ignored. The cell then lists any leftover indexers and indexes that start with your prefix, because the generated objects are separate Search objects. Delete leftovers in the portal or with notebook 03/04's patterns. It does not remove uploaded blobs, SharePoint files, the app registration or Azure infrastructure.

Clear notebook outputs before sharing; they can contain internal document text and web results.

In [ ]:
CLEAN_UP = False
if CLEAN_UP:
    for collection, name in [
        ('knowledgebases', knowledge_base_name), ('knowledgesources', webiq_source_name),
        ('knowledgesources', sharepoint_source_name), ('knowledgesources', blob_source_name),
    ]:
        response = requests.delete(
            f'{search_endpoint}/{collection}/{name}?api-version={api_version}',
            headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
        )
        if response.status_code != 404:
            response.raise_for_status()
        print('Deleted or already absent:', name)
    for collection in ('indexers', 'indexes'):
        response = requests.get(
            f'{search_endpoint}/{collection}?api-version={api_version}&$select=name',
            headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
        )
        response.raise_for_status()
        leftovers = [item['name'] for item in response.json()['value'] if item['name'].startswith(f'{prefix}-kb-')]
        print(f'Remaining {collection} with your prefix:', leftovers or 'none')
else:
    print('Cleanup disabled.')

## References

- [What is Foundry IQ?](https://learn.microsoft.com/azure/foundry/agents/concepts/what-is-foundry-iq)
- [Knowledge sources overview](https://learn.microsoft.com/azure/search/agentic-knowledge-source-overview)
- [Blob knowledge source](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-blob)
- [Indexed SharePoint knowledge source](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-sharepoint-indexed)
- [Remote SharePoint knowledge source](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-sharepoint-remote)
- [MCP server knowledge source](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-mcp-server)
- [Create a knowledge base](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-create-knowledge-base)
- [Retrieve from a knowledge base](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve)

Preview APIs, model availability and Microsoft 365 eligibility can change.